# 单节点串行中断
和之前并行中断一样的

In [1]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

class OverAllState(TypedDict):
    username: str # 姓名
    age: int # 年龄
    gender: Literal["male", "female"] # 性别

# 之前是两个同一个超步的节点同时中断， 这里是一个个中断
def get_info_node(state: OverAllState) -> OverAllState:
    username = interrupt("请输入您的用户名：")
    age = interrupt("请输入您的年龄：")
    gender = interrupt("请输入您的性别：(male/female)") # 一个节点三个中断

    return {
        "username": username,
        "age": age,
        "gender": gender
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("get_info_node", get_info_node)
builder.add_edge(START, "get_info_node")
builder.add_edge("get_info_node", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "seq_interrupt_test"}}
username_interrupted_res = graph.invoke({}, config=config) 
print('=' * 30, '-> username_interrupted_res <-', '=' * 30)
print(username_interrupted_res)

user_name = input("请输入您的用户名：") # 拿到中断，用户输入后恢复
age_interrupted_res = graph.invoke(Command(resume=user_name), config=config)
print('=' * 30, '-> age_interrupted_res <-', '=' * 30)
print(age_interrupted_res)

user_age = input("请输入您的年龄：") # 拿到中断，用户输入后恢复
gender_interrupted_res = graph.invoke(Command(resume=int(user_age)), config=config)
print('=' * 30, '-> gender_interrupted_res <-', '=' * 30)
print(gender_interrupted_res)

user_gender = input("请输入您的性别：(male/female): ")
resumed_res = graph.invoke(Command(resume=user_gender), config=config)
print('=' * 30, '-> resumed_res <-', '=' * 30)
print(resumed_res)

============================== -> username_interrupted_res <- ==============================
{'__interrupt__': [Interrupt(value='请输入您的用户名：', id='d00f2a57a0ba4dd015941fc8c5671760')]}
============================== -> age_interrupted_res <- ==============================
{'__interrupt__': [Interrupt(value='请输入您的年龄：', id='d00f2a57a0ba4dd015941fc8c5671760')]}
============================== -> gender_interrupted_res <- ==============================
{'__interrupt__': [Interrupt(value='请输入您的性别：(male/female)', id='d00f2a57a0ba4dd015941fc8c5671760')]}
============================== -> resumed_res <- ==============================
{'username': '小王', 'age': 12, 'gender': 'male'}


中断恢复时，整个被中断的节点函数都会重新运行。

单个节点中串行地多次调用 **`interrupt()`** 函数时，检查点存储器会记录历史的 **`resume`** 信息，**`LangGraph`** 运行时会读取这些信息，并在 **`interrupt()`** 函数中维护索引，按照节点内调用 **`interrupt()`** 函数的顺序，逐个取出历史 **`resume`** 的值，并将它们作为 **`interrupt()`** 函数的返回值。

所以，已经被恢复的 **`interrupt()`** 不会被重复触发。并且，由此可以推断，我们要保证历史 **`resume`** 可以被正确应用，就应保证中断恢复前后的多次 **`interrupt()`** 相对顺序保持不变。

当历史 **`resume`** 耗尽后，本次恢复运行时传入的 **`resume`** 会作为本次中断的返回值，然后节点函数继续运行。

所有中断都触发并恢复后，计算图正常结束。




这里有点特殊的说明，前面说恢复，是恢复节点， 但是在这里像恢复特定的中断， 有点像再用户又再年龄，又在性别，  但是本质都是再get_info_node

后面问用户名之后， 不再问年龄和性别了， 因为resume里面会有历史记录，resmue里面有用户名，所以第二次中断，再用节点函数的时候， 有信息了， 再填入，  没能力判断哪一行，挨个去填，